In [3]:
from nba_api.stats.static import teams
from nba_api.stats.endpoints import leaguegamefinder
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import time

In [4]:
# !pip install nba_api

In [5]:
nba_teams = teams.get_teams()
team_abbr_id = {team["abbreviation"]: team["id"] for team in nba_teams}
all_games = pd.DataFrame()

In [6]:
for team in nba_teams:
  team_id = team["id"]
  game_finder = leaguegamefinder.LeagueGameFinder(team_id_nullable=team_id, season_type_nullable="Regular Season")
  games = game_finder.get_data_frames()[0]
  all_games = pd.concat([all_games, games], ignore_index=True)
  time.sleep(0.5)

In [7]:
games_modern = all_games[
    (all_games.SEASON_ID.str[-4:].isin(["2021", "2022", "2023", "2024", "2025"]))
].copy()

In [8]:
games_modern["GAME_DATE"] = pd.to_datetime(games_modern["GAME_DATE"])
games_modern.sort_values(by="GAME_DATE", inplace=True)
games_modern["WIN"] = games_modern["WL"].apply(lambda x: 1 if x == "W" else 0)

games_modern["MIN"] = games_modern["MIN"].astype(float)  # minutes
games_modern["PTS"] = games_modern["PTS"].astype(float)  # points
games_modern["FGM"] = games_modern["FGM"].astype(float)  # field goals made
games_modern["FGA"] = games_modern["FGA"].astype(float)  # field goals attempted
games_modern["FG3M"] = games_modern["FG3M"].astype(float)  # 3s made
games_modern["FG3A"] = games_modern["FG3A"].astype(float)  # 3s attempted
games_modern["FTM"] = games_modern["FTM"].astype(float)  # free throws made
games_modern["FTA"] = games_modern["FTA"].astype(float)  # free throws attempted
games_modern["REB"] = games_modern["REB"].astype(float)  # rebounds
games_modern["OREB"] = games_modern["OREB"].astype(float)  # offensive rebounds
games_modern["DREB"] = games_modern["DREB"].astype(float)  # defensive rebounds
games_modern["AST"] = games_modern["AST"].astype(float)  # assists
games_modern["BLK"] = games_modern["BLK"].astype(float)  # blocks
games_modern["TOV"] = games_modern["TOV"].astype(float)  # turnovers
games_modern["PF"] = games_modern["PF"].astype(float)  # personal fouls

In [9]:
def get_opponent_id(matchup, team_abbr_to_id, team_id):
  if "@" in matchup:
    opponent_abbr = matchup.split(" @ ")[-1]
  else:
    opponent_abbr = matchup.split(" vs. ")[-1]
  return team_abbr_to_id.get(opponent_abbr, team_id)

games_modern["OPP_TEAM_ID"] = games_modern.apply(lambda row: get_opponent_id(row["MATCHUP"], team_abbr_id, row["TEAM_ID"]), axis=1)

In [10]:
games_modern["HGA"] = games_modern["MATCHUP"].apply(lambda x: 0 if "@" in x else 1)
games_modern["LAST_GAME_OUTCOME"] = (games_modern.groupby("TEAM_ID")["WIN"].shift(1).fillna(0))
games_modern["EFG%"] = (games_modern["FGM"] + (0.5 * games_modern["FG3M"])) / games_modern["FGA"]
games_modern["TOV%"] = games_modern["TOV"] / (games_modern["FGA"] + 0.44 * games_modern["FTA"] + games_modern["TOV"])
games_modern["FTR"] = games_modern["FTA"] / games_modern["FGA"]
games_modern["TS%"] = games_modern["PTS"] / (2 * (games_modern["FGA"] + (0.44 * games_modern["FTA"])))

In [11]:
stats_to_average = ["PTS", "OREB", "DREB", "REB", "AST", "STL", "BLK", "TOV", "EFG%", "TOV%", "FTR", "TS%"]

for stat in stats_to_average:
    games_modern[f"{stat}_5G_AVG"] = (games_modern.groupby("TEAM_ID")[stat].transform(lambda x: x.shift(1).rolling(window=5, min_periods=1).mean()))

games_modern.dropna(subset=[f"{stat}_5G_AVG" for stat in stats_to_average], inplace=True)

In [12]:
le = LabelEncoder()
games_modern["TEAM_ID"] = le.fit_transform(games_modern["TEAM_ID"])
games_modern["OPP_TEAM_ID"] = le.fit_transform(games_modern["OPP_TEAM_ID"])
games_modern["GAME_ID"] = le.fit_transform(games_modern["GAME_ID"])

In [13]:
features = ["TEAM_ID", "OPP_TEAM_ID", "PTS", "OREB", "DREB", "REB", "AST", "STL", "BLK", "TOV", "EFG%", "TOV%", "FTR", "TS%", "HGA", "LAST_GAME_OUTCOME",]
X = games_modern[features]
y = games_modern["WIN"]

In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [15]:
rf_model = RandomForestClassifier(n_estimators=175, max_depth=25, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.8338842975206612
